# Predicting Serum Estradiol Concentrations with Machine Learning

**An exploratory analysis of NHANES 2021–2023 data**

## Research question
How accurately can serum estradiol (pg/mL) be predicted from demographic, anthropometric and reproductive hormone measurements?

This notebook investigates distributions, statistical associations, regression models and the biological context of prediction errors. It is an **exploratory research portfolio**, not a clinically validated diagnostic tool.

**Data source:** [CDC NHANES 2021–2023](https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?Cycle=2021-2023). Data are downloaded from CDC public XPT files at runtime.

**Reproduction:** Run all cells in order in a fresh Python environment with the packages listed in `requirements.txt`. Runtime depends on network access to CDC files.

## 1. Data acquisition and preparation

The hormone, demographic and body data were loaded and merged on the participant identifier (`SEQN`). The exploratory cohort was restricted to adults aged 20 years or older.

In [ ]:

# STEP 1: Import the necessary libraries

import pandas as pd
import urllib.request

# STEP 2: Specify the official CDC dataset location

url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/TST_L.XPT"

# STEP 3: Download the dataset into Google Colab

urllib.request.urlretrieve(url, "TST_L.XPT")

# STEP 4: Load the downloaded dataset into Python

hormones = pd.read_sas("TST_L.XPT", format="xport")

# STEP 5: Confirm that the download worked

print("Dataset downloaded successfully!")

print("Number of rows and columns:", hormones.shape)

# Display the first five rows

display(hormones.head())


In [ ]:

# Download the NHANES demographics dataset

demo_url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.XPT"

demographics = pd.read_sas(
    demo_url,
    format="xport"
)

print("Demographics dataset downloaded!")

print("Rows and columns:", demographics.shape)

# Display the first five rows

display(demographics.head())


In [ ]:

# Download the NHANES body measurements dataset

bmx_url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.XPT"

body_measurements = pd.read_sas(
    bmx_url,
    format="xport"
)

print("Body measurements dataset downloaded!")

print("Rows and columns:", body_measurements.shape)

# Display the first five rows

display(body_measurements.head())


In [ ]:

# Merge hormone data with demographic data

combined_data = pd.merge(
    hormones,
    demographics[["SEQN", "RIAGENDR", "RIDAGEYR"]],
    on="SEQN",
    how="left"
)

# Add BMI measurements

combined_data = pd.merge(
    combined_data,
    body_measurements[["SEQN", "BMXBMI"]],
    on="SEQN",
    how="left"
)

print("Datasets successfully merged!")

print("Combined dataset dimensions:")
print(combined_data.shape)

# Display the first five rows

display(combined_data.head())


In [ ]:

# Create an adult-only dataset

adult_data = combined_data[
    combined_data["RIDAGEYR"] >= 20
].copy()

print("Adult dataset created!")

print("Number of adult participants:", len(adult_data))

# Display the first five rows

display(adult_data.head())


In [ ]:

# Check missing values in our important variables

important_variables = [
    "LBXEST",
    "RIAGENDR",
    "RIDAGEYR",
    "BMXBMI"
]

print("MISSING DATA SUMMARY")
print("--------------------")

display(
    adult_data[important_variables].isnull().sum()
)

# Check the age range

print("\nAGE RANGE")

print("Minimum age:", adult_data["RIDAGEYR"].min())
print("Maximum age:", adult_data["RIDAGEYR"].max())

# Check gender counts

print("\nGENDER DISTRIBUTION")

print(
    adult_data["RIAGENDR"].value_counts()
)


## 2. Exploratory data analysis

Missingness, the right-skewed estradiol distribution and patterns by sex, age and BMI were inspected. The plots and descriptive statistics are unweighted.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:

# Count missing values in every column

missing_values = hormones.isnull().sum()

# Calculate the percentage of missing values

missing_percentage = (
    hormones.isnull().mean() * 100
)

# Combine the results into a table

missing_summary = pd.DataFrame({
    "Missing values": missing_values,
    "Missing percentage (%)": missing_percentage
})

# Sort from most missing to least missing

missing_summary = missing_summary.sort_values(
    by="Missing percentage (%)",
    ascending=False
)

display(missing_summary)


In [ ]:

# Investigate estradiol concentrations in more detail

print("ESTRADIOL DISTRIBUTION ANALYSIS")
print("--------------------------------")

print("Minimum:", hormones["LBXEST"].min())
print("Maximum:", hormones["LBXEST"].max())
print("Median:", hormones["LBXEST"].median())
print("Mean:", hormones["LBXEST"].mean())

print("\nSelected percentiles:")

print(hormones["LBXEST"].quantile(
    [0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
))


In [ ]:

# Calculate the 99th percentile
upper_limit = hormones["LBXEST"].quantile(0.99)

# Select measurements within the plotting range
# This only filters the graph, not our original dataset

estradiol_plot = hormones.loc[
    hormones["LBXEST"].between(0, upper_limit),
    "LBXEST"
].dropna()

# Create the improved histogram

plt.figure(figsize=(10, 6))

sns.histplot(
    x=estradiol_plot,
    bins=40,
    kde=True
)

plt.title("Distribution of Serum Estradiol Concentrations (Central 99%)")
plt.xlabel("Estradiol concentration (pg/mL)")
plt.ylabel("Frequency")

plt.tight_layout()

# Save the corrected graph
plt.savefig(
    "estradiol_distribution_zoomed.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# Summary statistics for estradiol

print("MEAN:", hormones["LBXEST"].mean())
print("MEDIAN:", hormones["LBXEST"].median())
print("STANDARD DEVIATION:", hormones["LBXEST"].std())

print("\nPERCENTILES:")

display(
    hormones["LBXEST"].quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
    )
)


In [ ]:

# Compare estradiol concentrations between sexes

sex_summary = adult_data.groupby("RIAGENDR")["LBXEST"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)

# Replace numerical codes with readable labels

sex_summary.index = sex_summary.index.map({
    1.0: "Male",
    2.0: "Female"
})

# Display the results

print("ESTRADIOL STATISTICS BY SEX")
print("---------------------------")

display(sex_summary.round(2))


In [ ]:

# Prepare data for visualisation

plot_data = adult_data.dropna(
    subset=["LBXEST", "RIAGENDR"]
).copy()

# Convert numerical sex codes into labels

plot_data["Sex"] = plot_data["RIAGENDR"].map({
    1.0: "Male",
    2.0: "Female"
})

# Create boxplot

plt.figure(figsize=(9, 6))

sns.boxplot(
    data=plot_data,
    x="Sex",
    y="LBXEST"
)

plt.yscale("log")

plt.title("Serum Estradiol Concentrations by Sex")
plt.xlabel("Sex")
plt.ylabel("Estradiol concentration (pg/mL, logarithmic scale)")

plt.tight_layout()

plt.savefig(
    "estradiol_by_sex.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# Create a scatterplot of age against estradiol

age_plot_data = adult_data.dropna(
    subset=["RIDAGEYR", "LBXEST"]
)

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=age_plot_data,
    x="RIDAGEYR",
    y="LBXEST",
    alpha=0.35,
    s=20
)

plt.yscale("log")

plt.title("Relationship Between Age and Estradiol Concentration")
plt.xlabel("Age (years)")
plt.ylabel("Estradiol concentration (pg/mL, logarithmic scale)")

plt.tight_layout()

plt.savefig(
    "estradiol_vs_age.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# Import Spearman correlation

from scipy.stats import spearmanr

# Separate participants by sex

male_data = adult_data[
    adult_data["RIAGENDR"] == 1
].dropna(subset=["RIDAGEYR", "LBXEST"])

female_data = adult_data[
    adult_data["RIAGENDR"] == 2
].dropna(subset=["RIDAGEYR", "LBXEST"])

# Calculate correlations

male_corr, male_p = spearmanr(
    male_data["RIDAGEYR"],
    male_data["LBXEST"]
)

female_corr, female_p = spearmanr(
    female_data["RIDAGEYR"],
    female_data["LBXEST"]
)

# Display results

print("AGE AND ESTRADIOL CORRELATION")
print("----------------------------")

print("\nMALES")
print("Spearman correlation:", round(male_corr, 3))
print("P-value:", male_p)

print("\nFEMALES")
print("Spearman correlation:", round(female_corr, 3))
print("P-value:", female_p)


In [ ]:

# Create separate scatterplots for males and females

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Male scatterplot

sns.scatterplot(
    data=male_data,
    x="RIDAGEYR",
    y="LBXEST",
    alpha=0.4,
    ax=axes[0]
)

axes[0].set_yscale("log")
axes[0].set_title("Male Estradiol Concentrations")
axes[0].set_xlabel("Age (years)")
axes[0].set_ylabel("Estradiol (pg/mL)")

# Female scatterplot

sns.scatterplot(
    data=female_data,
    x="RIDAGEYR",
    y="LBXEST",
    alpha=0.4,
    ax=axes[1]
)

axes[1].set_yscale("log")
axes[1].set_title("Female Estradiol Concentrations")
axes[1].set_xlabel("Age (years)")
axes[1].set_ylabel("Estradiol (pg/mL)")

plt.suptitle(
    "Relationship Between Age and Estradiol by Sex",
    fontsize=15
)

plt.tight_layout()

plt.savefig(
    "estradiol_age_by_sex.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# Investigate the relationship between BMI and estradiol

from scipy.stats import spearmanr

# Prepare data for analysis

bmi_data = adult_data.dropna(
    subset=["BMXBMI", "LBXEST", "RIAGENDR"]
).copy()

# Separate males and females

male_bmi = bmi_data[bmi_data["RIAGENDR"] == 1]

female_bmi = bmi_data[bmi_data["RIAGENDR"] == 2]

# Calculate Spearman correlations

male_bmi_corr, male_bmi_p = spearmanr(
    male_bmi["BMXBMI"],
    male_bmi["LBXEST"]
)

female_bmi_corr, female_bmi_p = spearmanr(
    female_bmi["BMXBMI"],
    female_bmi["LBXEST"]
)

# Display results

print("BMI AND ESTRADIOL CORRELATION")
print("----------------------------")

print("\nMALES")
print("Participants:", len(male_bmi))
print("Spearman correlation:", round(male_bmi_corr, 3))
print("P-value:", male_bmi_p)

print("\nFEMALES")
print("Participants:", len(female_bmi))
print("Spearman correlation:", round(female_bmi_corr, 3))
print("P-value:", female_bmi_p)


In [ ]:

# Create BMI versus estradiol scatterplots

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Male participants

sns.scatterplot(
    data=male_bmi,
    x="BMXBMI",
    y="LBXEST",
    alpha=0.4,
    ax=axes[0]
)

axes[0].set_yscale("log")
axes[0].set_title("Male Participants")
axes[0].set_xlabel("BMI (kg/m²)")
axes[0].set_ylabel("Estradiol (pg/mL)")

# Female participants

sns.scatterplot(
    data=female_bmi,
    x="BMXBMI",
    y="LBXEST",
    alpha=0.4,
    ax=axes[1]
)

axes[1].set_yscale("log")
axes[1].set_title("Female Participants")
axes[1].set_xlabel("BMI (kg/m²)")
axes[1].set_ylabel("Estradiol (pg/mL)")

plt.suptitle(
    "Relationship Between BMI and Estradiol by Sex",
    fontsize=15
)

plt.tight_layout()

plt.savefig(
    "estradiol_vs_bmi.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# Summarise BMI measurements by sex

bmi_summary = bmi_data.groupby("RIAGENDR")["BMXBMI"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)

bmi_summary.index = bmi_summary.index.map({
    1.0: "Male",
    2.0: "Female"
})

print("BMI SUMMARY BY SEX")

display(bmi_summary.round(2))


## 3. Statistical associations and assay considerations

Spearman correlations were used to describe monotonic associations, while the Mann–Whitney test compared distributions. Because NHANES has a complex sampling design, these unweighted inferential tests were not interpreted as nationally representative population estimates.

In [ ]:

# Import statistical test

from scipy.stats import mannwhitneyu

# Extract estradiol measurements

male_estradiol = adult_data.loc[
    adult_data["RIAGENDR"] == 1,
    "LBXEST"
].dropna()

female_estradiol = adult_data.loc[
    adult_data["RIAGENDR"] == 2,
    "LBXEST"
].dropna()

# Perform Mann-Whitney U test

u_statistic, p_value = mannwhitneyu(
    male_estradiol,
    female_estradiol,
    alternative="two-sided"
)

# Display results

print("MANN-WHITNEY U TEST")
print("-------------------")

print("U statistic:", u_statistic)
print("P-value:", p_value)

if p_value < 0.05:
    print("Result: Statistically significant difference")
else:
    print("Result: No statistically significant difference")


In [ ]:

# Compare median estradiol concentrations

male_median = male_estradiol.median()
female_median = female_estradiol.median()

median_difference = female_median - male_median

print("MALE MEDIAN:", male_median)
print("FEMALE MEDIAN:", female_median)

print(
    "Female minus male median difference:",
    median_difference
)


In [ ]:

# Investigate estradiol detection-limit indicators

print("ESTRADIOL DETECTION LIMIT ANALYSIS")
print("----------------------------------")

print("\nDetection-limit indicator counts:")

print(
    adult_data["LBDESTLC"].value_counts(
        dropna=False
    )
)

# Compare estradiol measurements by detection status

print("\nEstradiol concentrations by detection status:")

display(
    adult_data.groupby("LBDESTLC")["LBXEST"].agg(
        ["count", "min", "median", "max"]
    )
)


In [ ]:

# Examine the 15 highest estradiol concentrations

highest_estradiol = adult_data[
    ["SEQN", "RIAGENDR", "RIDAGEYR",
     "BMXBMI", "LBXEST", "LBDESTLC"]
].sort_values(
    by="LBXEST",
    ascending=False
).head(15)

print("15 HIGHEST ESTRADIOL MEASUREMENTS")

display(highest_estradiol)


In [ ]:

# Inspect potential hormone predictor columns

potential_predictors = [
    "LBXFSH",
    "LBXLUH",
    "LBXPG4",
    "LBXTST",
    "LBXSHBG"
]

for variable in potential_predictors:
    if variable in adult_data.columns:
        missing = adult_data[variable].isna().sum()
        print(variable, "- Missing values:", missing)
    else:
        print(variable, "- Column not found")


## 4. Modelling dataset and train/test split

Eight predictors were used. Observations missing any required measurement were excluded alongside estradiol results below the assay detection limit. This complete-case approach limits generalisability. The target was transformed using `log1p`; the split is stratified by recorded sex.

In [ ]:

# Select variables for machine learning

model_variables = [
    "RIAGENDR",
    "RIDAGEYR",
    "BMXBMI",
    "LBXFSH",
    "LBXLUH",
    "LBXPG4",
    "LBXTST",
    "LBXSHBG",
    "LBXEST",
    "LBDESTLC"
]

# Create a separate modelling dataset

model_data = adult_data[model_variables].copy()

print("Original adult participants:", len(adult_data))

print("Modelling dataset dimensions:", model_data.shape)

display(model_data.head())


In [ ]:

# Count participants with complete measurements

complete_data = model_data.dropna(
    subset=[
        "RIAGENDR",
        "RIDAGEYR",
        "BMXBMI",
        "LBXFSH",
        "LBXLUH",
        "LBXPG4",
        "LBXTST",
        "LBXSHBG",
        "LBXEST"
    ]
).copy()

print("DATA COMPLETENESS")
print("-----------------")

print("Original adult participants:", len(model_data))

print("Complete observations:", len(complete_data))

print(
    "Participants excluded due to missing values:",
    len(model_data) - len(complete_data)
)

print(
    "Percentage retained:",
    round(len(complete_data) / len(model_data) * 100, 2)
)


In [ ]:

# Check for duplicate participant IDs

print("Duplicate participant IDs:")

print(adult_data["SEQN"].duplicated().sum())


In [ ]:

# Select biological variables

correlation_variables = [
    "RIDAGEYR",
    "BMXBMI",
    "LBXFSH",
    "LBXLUH",
    "LBXPG4",
    "LBXTST",
    "LBXSHBG",
    "LBXEST"
]

# Calculate Spearman correlations

correlation_matrix = adult_data[
    correlation_variables
].corr(method="spearman")

# Create heatmap

plt.figure(figsize=(10, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)

plt.title("Spearman Correlations Between Biological Variables")

plt.tight_layout()

plt.savefig(
    "hormone_correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# Create the dataset for our first predictive model

# Start with our complete observations
ml_data = complete_data.copy()

# Keep estradiol results at or above the detection limit
ml_data = ml_data[
    ml_data["LBDESTLC"] < 0.5
].copy()

print("MACHINE LEARNING DATASET")
print("------------------------")

print("Participants available:", len(ml_data))

print("Missing values:", ml_data.isnull().sum().sum())


In [ ]:

# Select our predictor variables

features = [
    "RIAGENDR",
    "RIDAGEYR",
    "BMXBMI",
    "LBXFSH",
    "LBXLUH",
    "LBXPG4",
    "LBXTST",
    "LBXSHBG"
]

# X contains the predictors
X = ml_data[features].copy()

# y contains the estradiol measurements
y = ml_data["LBXEST"].copy()

print("Number of predictors:", X.shape[1])
print("Number of participants:", X.shape[0])

print("\nTarget variable:")
print(y.name)


In [ ]:

# Import the training and testing function

from sklearn.model_selection import train_test_split

# Split the dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=X["RIAGENDR"]
)

print("TRAINING AND TESTING DATA")
print("-------------------------")

print("Training participants:", len(X_train))
print("Testing participants:", len(X_test))


In [ ]:
import numpy as np
# Import machine learning tools

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

# Transform the target to reduce skewness

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# Baseline model
baseline = DummyRegressor(strategy="median")

baseline.fit(X_train, y_train_log)

# Ridge regression model
ridge_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

ridge_model.fit(X_train, y_train_log)

print("Both models trained successfully!")


In [ ]:

# Import Random Forest

from sklearn.ensemble import RandomForestRegressor

# Create the model

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

# Train the model using the same training data

rf_model.fit(X_train, y_train_log)

print("Random Forest trained successfully!")


In [ ]:

# Import cross-validation tools

import numpy as np

from sklearn.model_selection import KFold, cross_validate

# Create five cross-validation folds

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluate Random Forest on training data only

cv_results = cross_validate(
    rf_model,
    X_train,
    y_train_log,
    cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    },
    n_jobs=-1
)

# Convert negative MAE scores to positive values

cv_mae = -cv_results["test_mae"]
cv_r2 = cv_results["test_r2"]

print("RANDOM FOREST CROSS-VALIDATION")
print("--------------------------------")

print("MAE for each fold:", np.round(cv_mae, 3))
print("R-squared for each fold:", np.round(cv_r2, 3))

print("\nAverage MAE:", round(cv_mae.mean(), 3))
print("MAE standard deviation:", round(cv_mae.std(), 3))

print("\nAverage R-squared:", round(cv_r2.mean(), 3))
print("R-squared standard deviation:", round(cv_r2.std(), 3))


In [ ]:

# STAGE 17.1 - Train Gradient Boosting

from sklearn.ensemble import HistGradientBoostingRegressor

# Create the model

gb_model = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=15,
    min_samples_leaf=20,
    random_state=42
)

# Train using our existing training data

gb_model.fit(X_train, y_train_log)

print("Gradient Boosting trained successfully!")


In [ ]:

# STAGE 17.2 - Cross-validate Gradient Boosting

from sklearn.model_selection import cross_validate

gb_cv_results = cross_validate(
    gb_model,
    X_train,
    y_train_log,
    cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    },
    n_jobs=-1
)

gb_cv_mae = -gb_cv_results["test_mae"]
gb_cv_r2 = gb_cv_results["test_r2"]

print("GRADIENT BOOSTING CROSS-VALIDATION")
print("----------------------------------")

print("MAE for each fold:", np.round(gb_cv_mae, 3))
print("R-squared for each fold:", np.round(gb_cv_r2, 3))

print("\nAverage MAE:", round(gb_cv_mae.mean(), 3))
print("MAE standard deviation:", round(gb_cv_mae.std(), 3))

print("\nAverage R-squared:", round(gb_cv_r2.mean(), 3))
print("R-squared standard deviation:", round(gb_cv_r2.std(), 3))


In [ ]:

# STAGE 17.3 - Compare cross-validation performance

cv_comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Gradient Boosting"
    ],
    "Mean MAE (log scale)": [
        cv_mae.mean(),
        gb_cv_mae.mean()
    ],
    "MAE Standard Deviation": [
        cv_mae.std(),
        gb_cv_mae.std()
    ],
    "Mean R-squared": [
        cv_r2.mean(),
        gb_cv_r2.mean()
    ],
    "R-squared Standard Deviation": [
        cv_r2.std(),
        gb_cv_r2.std()
    ]
})

print("CROSS-VALIDATION MODEL COMPARISON")
print("---------------------------------")

display(cv_comparison.round(3))


## 5. Exploratory test-set performance

The same test split was inspected repeatedly while developing the project, so **test results are exploratory rather than an untouched final validation**. Cross-validation on training observations was used for the primary algorithm comparison.

In [ ]:

# Generate predictions on the test data

baseline_pred = baseline.predict(X_test)
ridge_pred = ridge_model.predict(X_test)

# Evaluate performance on the log scale

print("MODEL PERFORMANCE (LOG SCALE)")
print("-----------------------------")

for name, predictions in [
    ("Baseline", baseline_pred),
    ("Ridge Regression", ridge_pred)
]:
    mae = mean_absolute_error(y_test_log, predictions)
    r2 = r2_score(y_test_log, predictions)

    print("\n", name)
    print("MAE:", round(mae, 3))
    print("R-squared:", round(r2, 3))

# Convert predictions back to pg/mL

ridge_pred_pg = np.expm1(ridge_pred)

print("\nRIDGE REGRESSION (ORIGINAL SCALE)")
print("--------------------------------")

print(
    "MAE (pg/mL):",
    round(mean_absolute_error(y_test, ridge_pred_pg), 2)
)


In [ ]:

# Generate Random Forest predictions

rf_pred_log = rf_model.predict(X_test)

# Calculate performance on log scale

rf_mae = mean_absolute_error(
    y_test_log,
    rf_pred_log
)

rf_r2 = r2_score(
    y_test_log,
    rf_pred_log
)

# Convert predictions back to pg/mL

rf_pred_pg = np.expm1(rf_pred_log)

rf_mae_pg = mean_absolute_error(
    y_test,
    rf_pred_pg
)

print("RANDOM FOREST PERFORMANCE")
print("-------------------------")

print("MAE (log scale):", round(rf_mae, 3))
print("R-squared (log scale):", round(rf_r2, 3))
print("MAE (pg/mL):", round(rf_mae_pg, 2))


In [ ]:

# Compare model performance

import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Ridge Regression",
        "Random Forest"
    ],
    "MAE (log scale)": [
        mean_absolute_error(y_test_log, baseline_pred),
        mean_absolute_error(y_test_log, ridge_pred),
        rf_mae
    ],
    "R-squared (log scale)": [
        r2_score(y_test_log, baseline_pred),
        r2_score(y_test_log, ridge_pred),
        rf_r2
    ]
})

display(comparison.round(3))


In [ ]:

# Calculate Random Forest feature importance

importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("RANDOM FOREST FEATURE IMPORTANCE")

display(importance.round(3))

# Visualise feature importance

plt.figure(figsize=(9, 6))

sns.barplot(
    data=importance,
    x="Importance",
    y="Feature"
)

plt.title("Random Forest Feature Importance")
plt.xlabel("Relative importance")
plt.ylabel("Predictor")

plt.tight_layout()

plt.savefig(
    "random_forest_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# Plot actual versus predicted estradiol

plt.figure(figsize=(8, 7))

plt.scatter(
    y_test,
    rf_pred_pg,
    alpha=0.4,
    s=20
)

# Reference line for perfect predictions

max_value = max(y_test.max(), rf_pred_pg.max())

plt.plot(
    [0, max_value],
    [0, max_value],
    linestyle="--",
    color="red",
    label="Perfect prediction"
)

plt.xscale("log")
plt.yscale("log")

plt.xlabel("Actual estradiol (pg/mL)")
plt.ylabel("Predicted estradiol (pg/mL)")

plt.title("Random Forest: Actual vs Predicted Estradiol")

plt.legend()
plt.tight_layout()

plt.savefig(
    "random_forest_predictions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# STAGE 14.1 - Calculate prediction errors

import pandas as pd
import numpy as np

# Create a table containing actual and predicted values

error_data = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": rf_pred_pg
})

# Calculate prediction errors

error_data["Error"] = (
    error_data["Predicted"] - error_data["Actual"]
)

error_data["Absolute_Error"] = (
    error_data["Error"].abs()
)

# Calculate absolute percentage error
# Only for positive actual concentrations

error_data["Percentage_Error"] = (
    error_data["Absolute_Error"]
    / error_data["Actual"]
) * 100

print("PREDICTION ERROR SUMMARY")
print("------------------------")

display(error_data.describe().round(2))


In [ ]:

# STAGE 14.2 - Find the largest prediction errors

largest_errors = error_data.sort_values(
    by="Absolute_Error",
    ascending=False
).head(10)

print("10 LARGEST PREDICTION ERRORS")
print("----------------------------")

display(largest_errors.round(2))


In [ ]:

# STAGE 14.3 - Compare errors by concentration range

# Define concentration categories

bins = [0, 10, 50, 200, np.inf]

labels = [
    "Low (<10)",
    "Moderate (10-<50)",
    "High (50-<200)",
    "Very high (200+)"
]

# Categorise participants

error_data["Concentration_Group"] = pd.cut(
    error_data["Actual"],
    bins=bins,
    labels=labels,
    right=False
)

# Calculate errors for each group

group_errors = error_data.groupby(
    "Concentration_Group",
    observed=True
).agg(
    Participants=("Actual", "count"),
    Mean_Absolute_Error=("Absolute_Error", "mean"),
    Median_Absolute_Error=("Absolute_Error", "median")
)

print("MODEL ACCURACY BY ESTRADIOL CONCENTRATION")
print("-----------------------------------------")

display(group_errors.round(2))


In [ ]:

# STAGE 14.4 - Visualise prediction errors

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=error_data,
    x="Concentration_Group",
    y="Absolute_Error",
    showfliers=False
)

plt.yscale("log")

plt.title(
    "Random Forest Prediction Errors by Estradiol Concentration"
)

plt.xlabel("Actual estradiol concentration (pg/mL)")
plt.ylabel("Absolute prediction error (pg/mL, log scale)")

plt.xticks(rotation=15)

plt.tight_layout()

plt.savefig(
    "prediction_errors_by_concentration.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# STAGE 14.5 - Calculate prediction bias

print("PREDICTION BIAS")
print("---------------")

print(
    "Mean prediction error:",
    round(error_data["Error"].mean(), 2),
    "pg/mL"
)

print(
    "Median prediction error:",
    round(error_data["Error"].median(), 2),
    "pg/mL"
)

print(
    "Percentage of predictions below actual values:",
    round((error_data["Error"] < 0).mean() * 100, 2),
    "%"
)


## 6. Pregnancy status and prediction errors

Pregnancy status was drawn from the demographics file and was missing or inapplicable for many participants. These missing entries have not and must not be interpreted as non-pregnant. Small subgroup sizes and different concentration distributions limit comparisons of absolute error.

In [ ]:

# STAGE 15.4 - Add pregnancy information

# Create a separate dataset so we don't overwrite adult_data

pregnancy_data = adult_data.merge(
    demographics[["SEQN", "RIDEXPRG"]],
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("PREGNANCY DATA MERGED")
print("---------------------")

print("Original participants:", len(adult_data))
print("Participants after merge:", len(pregnancy_data))

print("\nPregnancy status counts:")

print(
    pregnancy_data["RIDEXPRG"].value_counts(
        dropna=False
    )
)


In [ ]:

# STAGE 15.5 - Investigate high estradiol concentrations

highest_estradiol_pregnancy = pregnancy_data[
    [
        "SEQN",
        "RIAGENDR",
        "RIDAGEYR",
        "LBXEST",
        "RIDEXPRG"
    ]
].sort_values(
    by="LBXEST",
    ascending=False
).head(20)

print("20 HIGHEST ESTRADIOL MEASUREMENTS")
print("--------------------------------")

display(highest_estradiol_pregnancy)


In [ ]:

# STAGE 16.1 - Match prediction errors with pregnancy status

# Retrieve pregnancy status for each test participant
# using the original participant ID (SEQN)

test_pregnancy = pregnancy_data.set_index(
    "SEQN"
)["RIDEXPRG"]

# Retrieve SEQN values corresponding to the test observations
test_ids = adult_data.loc[X_test.index, "SEQN"]

# Create a new error table
pregnancy_errors = error_data.copy()

# Add participant IDs
pregnancy_errors["SEQN"] = test_ids.to_numpy()

# Match pregnancy status by participant ID
pregnancy_errors["Pregnancy_Status"] = (
    pregnancy_errors["SEQN"].map(test_pregnancy)
)

print("PREGNANCY STATUS IN TEST DATA")
print("-----------------------------")

print(
    pregnancy_errors["Pregnancy_Status"].value_counts(
        dropna=False
    )
)


In [ ]:

# STAGE 16.2 - Compare errors by pregnancy status

# Select participants with known pregnancy status

known_pregnancy = pregnancy_errors[
    pregnancy_errors["Pregnancy_Status"].isin([1, 2])
].copy()

# Convert numerical codes to readable labels

known_pregnancy["Pregnancy_Group"] = (
    known_pregnancy["Pregnancy_Status"].map({
        1: "Pregnant",
        2: "Not pregnant"
    })
)

# Calculate prediction errors for each group

pregnancy_comparison = known_pregnancy.groupby(
    "Pregnancy_Group"
).agg(
    Participants=("Actual", "count"),
    Mean_Absolute_Error=("Absolute_Error", "mean"),
    Median_Absolute_Error=("Absolute_Error", "median"),
    Mean_Actual_Estradiol=("Actual", "mean")
)

print("MODEL ACCURACY BY PREGNANCY STATUS")
print("----------------------------------")

display(pregnancy_comparison.round(2))


In [ ]:

# STAGE 16.3 - Largest errors and pregnancy status

print("15 LARGEST ERRORS WITH PREGNANCY STATUS")
print("---------------------------------------")

display(
    pregnancy_errors[
        [
            "SEQN",
            "Actual",
            "Predicted",
            "Absolute_Error",
            "Pregnancy_Status"
        ]
    ].sort_values(
        by="Absolute_Error",
        ascending=False
    ).head(15).round(2)
)


## 7. Final performance summary and sensitivity analysis

Log-scale and original-scale metrics were compared and the change in results across actual estradiol concentrations were inspected. Subgroup R² values can be negative when within-group variation is narrow. Subgroups have been defined retrospectively using observed outcomes.

In [ ]:

# STAGE 18.1 - Final Random Forest evaluation

from sklearn.metrics import mean_absolute_error, r2_score

# Predict estradiol on the log scale
final_predictions_log = rf_model.predict(X_test)

# Convert predictions back to pg/mL
final_predictions_pg = np.expm1(final_predictions_log)

# Calculate log-scale performance
final_mae_log = mean_absolute_error(
    y_test_log,
    final_predictions_log
)

final_r2_log = r2_score(
    y_test_log,
    final_predictions_log
)

# Calculate original-scale performance
final_mae_pg = mean_absolute_error(
    y_test,
    final_predictions_pg
)

# Calculate original-scale R-squared
final_r2_pg = r2_score(
    y_test,
    final_predictions_pg
)

print("FINAL RANDOM FOREST PERFORMANCE")
print("-------------------------------")

print("Training participants:", len(X_train))
print("Testing participants:", len(X_test))

print("\nLOG-TRANSFORMED ESTRADIOL")
print("MAE:", round(final_mae_log, 3))
print("R-squared:", round(final_r2_log, 3))

print("\nORIGINAL ESTRADIOL (pg/mL)")
print("MAE:", round(final_mae_pg, 2))
print("R-squared:", round(final_r2_pg, 3))


In [ ]:

# STAGE 18.2 - Create final model summary

final_results = pd.DataFrame({
    "Metric": [
        "Training participants",
        "Testing participants",
        "Cross-validation MAE (log)",
        "Cross-validation R-squared",
        "Test MAE (log)",
        "Test R-squared (log)",
        "Test MAE (pg/mL)",
        "Test R-squared (pg/mL)"
    ],
    "Value": [
        len(X_train),
        len(X_test),
        round(cv_mae.mean(), 3),
        round(cv_r2.mean(), 3),
        round(final_mae_log, 3),
        round(final_r2_log, 3),
        round(final_mae_pg, 2),
        round(final_r2_pg, 3)
    ]
})

display(final_results)

# Export results for the GitHub project
final_results.to_csv(
    "random_forest_results.csv",
    index=False
)

print("Results saved successfully!")


In [ ]:

# STAGE 19.1 - Model performance by estradiol range

from sklearn.metrics import mean_absolute_error, r2_score

# Create a table containing actual and predicted values

range_results = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": final_predictions_pg
})

# Define estradiol concentration groups

range_results["Estradiol_Range"] = pd.cut(
    range_results["Actual"],
    bins=[0, 10, 50, 200, float("inf")],
    labels=[
        "Below 10",
        "10 to <50",
        "50 to <200",
        "200 and above"
    ],
    right=False
)

# Calculate performance within each group

summary_rows = []

for group, data in range_results.groupby(
    "Estradiol_Range",
    observed=True
):

    mae = mean_absolute_error(
        data["Actual"],
        data["Predicted"]
    )

    r2 = (
        r2_score(data["Actual"], data["Predicted"])
        if len(data) >= 2 else np.nan
    )

    summary_rows.append({
        "Estradiol range": group,
        "Participants": len(data),
        "MAE (pg/mL)": round(mae, 2),
        "R-squared": round(r2, 3)
    })

range_performance = pd.DataFrame(summary_rows)

print("MODEL PERFORMANCE BY ESTRADIOL RANGE")
print("------------------------------------")

display(range_performance)


In [ ]:

# STAGE 19.2 - Sensitivity analysis

# Select test participants with estradiol below 200 pg/mL

lower_range = range_results[
    range_results["Actual"] < 200
]

# Calculate model performance

lower_mae = mean_absolute_error(
    lower_range["Actual"],
    lower_range["Predicted"]
)

lower_r2 = r2_score(
    lower_range["Actual"],
    lower_range["Predicted"]
)

print("SENSITIVITY ANALYSIS: ESTRADIOL BELOW 200 pg/mL")
print("------------------------------------------------")

print("Participants:", len(lower_range))
print("MAE:", round(lower_mae, 2), "pg/mL")
print("R-squared:", round(lower_r2, 3))

print("\nFULL TEST SET")
print("-------------")

print("Participants:", len(y_test))
print("MAE:", round(final_mae_pg, 2), "pg/mL")
print("R-squared:", round(final_r2_pg, 3))


## 8. Discussion

Random Forest achieved five-fold training-set cross-validation mean **R² = 0.788** and **MAE = 0.377** on the log-transformed target; Gradient Boosting performed similarly (R² = 0.790, MAE = 0.379). Random Forest was retained as the illustrative final model because its cross-validated MAE was marginally lower.

On the repeatedly inspected test split (n = 1,027), Random Forest achieved **log-scale R² = 0.741**, **original-scale R² = 0.899** and **original-scale MAE = 17.24 pg/mL**. The original-scale R² was sensitive to very high concentrations: among test observations below 200 pg/mL, R² was **0.677** and MAE **10.39 pg/mL**. These metrics assess different outcome distributions and should not be conflated.

All 20 highest observed estradiol measurements were from participants recorded as pregnant. In the test split, the six participants recorded as pregnant had higher mean absolute errors than the 181 recorded as not pregnant, but concentration ranges and sample sizes differed substantially. This does not establish pregnancy as an independent cause of model error.

### Limitations

- NHANES survey weights, strata and clusters were not incorporated; estimates and tests are not population-representative.
- Complete-case selection and exclusion of below-detection estradiol measurements may introduce selection bias.
- Pregnancy status was not a predictor and is missing or inapplicable for many participants; menstrual cycle, medication use and other relevant factors were not explicitly modelled.
- Other laboratory hormones are required as inputs, so the model does not replace estradiol testing.
- Predictors can be correlated; impurity-based feature importance is not causal evidence.
- The test set was examined repeatedly, and there was no independent external validation. The results are exploratory and **not appropriate for clinical decision-making**.

### Conclusion

Reproductive hormone and demographic measurements contained predictive information about estradiol, but prediction errors varied substantially with concentration. Combining statistical evaluation with biological interpretation exposed limitations that a single headline R² would obscure.
